# 06 – Ensemble Extension (leaderboard climb, optional)

**ENGE707 Assessment 3 – Facial Feature Classification Challenge (Kaggle)**

**Purpose.** Try to improve on the current best ensembles by adding **new, different backbones** (EfficientNetV2-S and
ConvNeXt-Tiny) trained with the same two-stage transfer-learning recipe, and averaging their predictions with the eight
models already trained (4 baseline from notebooks 03 and 03_rerun, 4 regularised from notebook 04). Nothing is retrained
from scratch: the existing models are reused through their saved probability files.

**Rules fixed in advance (same style as the report).**
1. Test images are used only to make predictions, never for training or pseudo-labelling.
2. A new submission file is written only if the ensemble has at most `MAX_VAL_ERRORS` validation errors (set in Step 4).
3. Validation errors are a noisy guide (about ±10 errors between runs, and validation runs about 1.8 points optimistic), so
   a result only counts as a real improvement if it is clearly better on validation **and** on the public leaderboard.
4. The report is updated only for a significant improvement (public score of about 0.982 or more).

**Inputs.** `../src/pipeline.py`, `../backup_run1`, `../backup_run2`, `../models/reg_*` (from notebooks 03, 03_rerun and 04).

**Outputs.** `../models/new_*` (probabilities and checkpoints) and, if the rule is met, `../submissions/submission_v5_ensemble.csv`.

**How to run.** From the `notebooks/` folder, after notebooks 03, 03_model_rerun and 04. Training takes roughly 10–15 minutes
per backbone on a GPU. **Not yet run: run it cell by cell and check the printed shapes and error counts.** If a backbone
fails to load (library version), skip that backbone and its cell.

## Step 1: Set up the data and helpers

Imports the shared pipeline, builds the training and validation datasets and defines a helper that counts validation errors.

In [ ]:
import sys; sys.path.append("../src")  # make ../src importable
import numpy as np, pandas as pd, tensorflow as tf  # numpy: arrays; pandas: tables; TensorFlow: deep learning
from tensorflow.keras import layers, models  # Keras building blocks
import pipeline  # shared data module (split, labels, tf.data pipelines)

SEED = 42  # fixed seed, as in the other notebooks
tf.keras.utils.set_random_seed(SEED)  # seeds Python, NumPy and TensorFlow together
y = pipeline.val_df["label"].values  # true validation classes
train_ds = pipeline.make_dataset(pipeline.train_df, shuffle=True, augment_data=True)  # training data: shuffled + augmented
val_ds = pipeline.make_dataset(pipeline.val_df, shuffle=False, augment_data=False)  # validation data: fixed order, no augmentation
errs = lambda q: int((q.argmax(axis=1) != y).sum())  # helper: validation errors of a probability table
print("Train:", len(pipeline.train_df), "| Val:", len(pipeline.val_df))  # expect 50928 and 12732

## Step 2: Define the two-stage training function

One function that trains any backbone with the report's recipe: stage 1 with the backbone frozen (5 epochs, learning rate 1e-3), then stage 2 fine-tuning the last 20% of the backbone's layers with BatchNorm frozen (up to 12 epochs, learning rate 1e-4, early stopping and learning-rate reduction). It saves the validation and test probabilities.

In [ ]:
def train_backbone(tag, make_base, seed=42, unfreeze_frac=0.2):  # two-stage training of one backbone; saves its probabilities
    tf.keras.utils.set_random_seed(seed)  # seed for this run
    base = make_base()  # the pretrained backbone (ImageNet weights, no classifier head)
    base.trainable = False  # stage 1: frozen backbone, train only the new head
    inputs = layers.Input(shape=(224, 224, 3))  # model input: 224 x 224 RGB, values in [0,1]
    x = layers.Rescaling(255.0)(inputs)  # these backbones expect 0-255 pixels (their own preprocessing is built in)
    x = base(x, training=False)  # frozen backbone in inference mode
    x = layers.GlobalAveragePooling2D()(x)  # average each feature map to one number
    x = layers.Dropout(0.3)(x)  # same dropout as the other models
    outputs = layers.Dense(8, activation="softmax")(x)  # 8 class probabilities
    model = models.Model(inputs, outputs)  # assemble the full model
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),  # stage 1 learning rate 1e-3
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])  # same loss and metric as notebook 03
    model.fit(train_ds, validation_data=val_ds, epochs=5, verbose=2)  # stage 1: 5 epochs

    base.trainable = True  # stage 2: allow the backbone to change...
    cut = int(len(base.layers) * (1 - unfreeze_frac))  # ...but only its last 20% of layers
    for i, layer in enumerate(base.layers):  # go through the backbone layer by layer
        layer.trainable = (i >= cut) and not isinstance(layer, layers.BatchNormalization)  # last layers only; BatchNorm stays frozen
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),  # ten times lower learning rate for fine-tuning
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])  # recompile after changing trainable
    cb = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),  # stop early, keep best weights
          tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=1, min_lr=1e-6, verbose=1)]  # lower the learning rate when stalled
    model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=cb, verbose=2)  # stage 2: up to 12 epochs

    pv = model.predict(val_ds, verbose=0)  # validation probabilities
    pt = model.predict(pipeline.make_test_dataset(), verbose=0)  # test probabilities (predictions only)
    np.save(f"../models/new_{tag}_val.npy", pv); np.save(f"../models/new_{tag}_test.npy", pt)  # save both for the ensembles
    print(f"{tag}: {errs(pv)} validation errors | accuracy {1 - errs(pv) / len(y):.4f}")  # result of this model
    return pv, pt  # hand the probabilities back

## Step 3: Train EfficientNetV2-S

Trains the first new backbone. Prints its validation errors (the existing single models have 59–75).

In [ ]:
from tensorflow.keras.applications import EfficientNetV2S  # a newer EfficientNet family member
pv_v2s, pt_v2s = train_backbone(  # train and evaluate EfficientNetV2-S
    "effv2s", lambda: EfficientNetV2S(include_top=False, weights="imagenet", input_shape=(224, 224, 3)))

## Step 4: Train ConvNeXt-Tiny

Trains the second new backbone, a different architecture family, which should add more diversity to the ensemble.

In [ ]:
from tensorflow.keras.applications import ConvNeXtTiny  # a modern convolutional network (different design from EfficientNet/ResNet)
pv_cnx, pt_cnx = train_backbone(  # train and evaluate ConvNeXt-Tiny
    "convnext", lambda: ConvNeXtTiny(include_top=False, weights="imagenet", input_shape=(224, 224, 3)))

## Step 5: Compare ensembles on the validation set

Loads the saved probabilities of the eight existing models and of whichever new backbones have been trained (so Steps 3 and 4 can be run in separate sessions), checks that the two known ensembles reproduce their earlier errors (35 and 45), and compares ensembles that include the new models. `MAX_VAL_ERRORS` is the submission rule.

In [ ]:
MAX_VAL_ERRORS = 30  # rule fixed in advance: write a submission only if the ensemble has at most this many validation errors

lv = lambda run, f: np.load(f"../{run}/models/{f}")  # helper: load a saved array from a backup folder
lm = lambda f: np.load(f"../models/{f}")  # helper: load a saved array from the models folder
old = {  # the 8 existing models: (validation probabilities, test probabilities)
    "base_eff1": (lv("backup_run1", "val_probs.npy"), lv("backup_run1", "test_probs.npy")),  # baseline EfficientNetB0, run 1
    "base_r501": (lv("backup_run1", "val_probs_r50.npy"), lv("backup_run1", "test_probs_r50.npy")),  # baseline ResNet50V2, run 1
    "base_eff2": (lv("backup_run2", "val_probs.npy"), lv("backup_run2", "test_probs.npy")),  # baseline EfficientNetB0, rerun
    "base_r502": (lv("backup_run2", "val_probs_r50.npy"), lv("backup_run2", "test_probs_r50.npy")),  # baseline ResNet50V2, rerun
    "reg_eff1": (lm("reg_val_probs.npy"), lm("reg_test_probs.npy")),  # regularised EfficientNetB0, seed 42
    "reg_eff2": (lm("reg2_val_probs.npy"), lm("reg2_test_probs.npy")),  # regularised EfficientNetB0, seed 7
    "reg_r501": (lm("reg_r50_val_probs.npy"), lm("reg_r50_test_probs.npy")),  # regularised ResNet50V2, seed 42
    "reg_r502": (lm("reg_r50b_val_probs.npy"), lm("reg_r50b_test_probs.npy")),  # regularised ResNet50V2, seed 7
}
import os  # os: check which saved files exist
new = {}  # the new models available so far: name -> (validation probabilities, test probabilities)
for tag in ("effv2s", "convnext"):  # the two new backbones
    if os.path.exists(f"../models/new_{tag}_val.npy"):  # only use a backbone that has been trained and saved
        new[tag] = (lm(f"new_{tag}_val.npy"), lm(f"new_{tag}_test.npy"))  # load its saved probabilities
print("New models available:", list(new))  # shows which backbones will be included
assert new, "Train at least one new backbone first (Step 3 or Step 4)"  # stop with a clear message if none exist yet
k = len(new)  # number of new models, used in the candidate names below
allm = {**old, **new}  # every available model
reg4 = ["reg_eff1", "reg_eff2", "reg_r501", "reg_r502"]  # the four regularised models (v4a)
six = ["base_eff1", "base_r501", "base_eff2", "base_r502", "reg_eff1", "reg_eff2"]  # the six models of v4b
candidates = {  # ensembles to compare, as lists of model names
    "v4a (4 regularised) [check: 35]": reg4,  # should reproduce 35 errors
    "v4b (6 models) [check: 45]": six,  # should reproduce 45 errors
    "all 8 existing": list(old),  # every existing model
    f"v4a + {k} new": reg4 + list(new),  # regularised four plus the two new backbones
    f"v4b + {k} new": six + list(new),  # six-model mix plus the two new backbones
    f"all 8 + {k} new": list(allm),  # everything
    "new models only": list(new),  # only the new backbones
}
results = {}  # name -> validation errors
for name, members in candidates.items():  # evaluate each candidate ensemble
    p = sum(allm[m][0] for m in members) / len(members)  # equal-weight average of the members' validation probabilities
    results[name] = errs(p)  # count its validation errors
    print(f"{name:34s} {len(members):2d} models | {results[name]:3d} errors | accuracy {1 - results[name] / len(y):.4f}")  # one line per ensemble

## Step 6: Write submission v5 if the rule is met

Takes the best ensemble that includes a new model and writes `submission_v5_ensemble.csv` only if its validation errors are within `MAX_VAL_ERRORS`. Upload it to Kaggle by hand. Do not change the report unless the public score improves significantly.

In [ ]:
best = min((n for n in candidates if "new" in n or "all 8 +" in n), key=lambda n: results[n])  # best candidate that includes a new model
print("Best candidate with a new model:", best, "->", results[best], "errors")  # show the choice
if results[best] <= MAX_VAL_ERRORS:  # only write a file if the pre-set rule is met
    members = candidates[best]  # members of the chosen ensemble
    t = sum(allm[m][1] for m in members) / len(members)  # equal-weight average of their test probabilities
    sub = pd.DataFrame({"image_id": pipeline.test_df["image_id"].values,  # test ids in Kaggle's order
                        "class": [pipeline.idx_to_class[i] for i in t.argmax(axis=1)]})  # predicted class names
    sub.to_csv("../submissions/submission_v5_ensemble.csv", index=False)  # write the submission file
    sample = pd.read_csv("../data/sample_submission.csv")  # Kaggle's template for the format check
    print("Saved. Shape:", sub.shape, "| columns match:", list(sub.columns) == list(sample.columns),  # shape and column check
          "| IDs match:", (sub["image_id"].values == sample["image_id"].values).all())  # id order check
else:  # rule not met
    print(f"Rule not met (more than {MAX_VAL_ERRORS} errors): no submission file written.")  # nothing is written